# Comparacao Sistematica de Modelos GARCH

Neste notebook, realizamos uma **comparacao sistematica** de todos os modelos GARCH
univariados disponiveis na biblioteca archbox.

**Modelos comparados:**
1. GARCH(1,1) - Bollerslev (1986)
2. EGARCH(1,1) - Nelson (1991)
3. GJR-GARCH(1,1) - Glosten, Jagannathan e Runkle (1993)
4. APARCH(1,1) - Ding, Granger e Engle (1993)
5. IGARCH(1,1) - Engle e Bollerslev (1986)
6. Component-GARCH - Engle e Lee (1999)

**Criterios de avaliacao:**
- Criterios de informacao (AIC, BIC, HQIC)
- Diagnosticos dos residuos (ARCH-LM, Ljung-Box)
- Previsao out-of-sample (rolling window)

**Conteudo:**
1. Estimando todos os modelos
2. Criterios de informacao
3. Diagnosticos dos residuos
4. Previsao out-of-sample
5. Ranking final
6. Conclusoes e recomendacoes praticas

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from utils.plot_helpers import plot_model_comparison

from archbox.diagnostics import arch_lm_test, ljung_box_squared
from archbox.models import APARCH, EGARCH, GARCH, GJRGARCH, IGARCH, ComponentGARCH

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Estimando todos os modelos

Vamos estimar os 6 modelos GARCH univariados no dataset do **S&P 500**.

Cada modelo captura aspectos diferentes da dinamica da volatilidade:

| Modelo | Caracteristica principal |
|--------|-------------------------|
| GARCH | Baseline simetrico |
| EGARCH | Assimetria via $\ln(\sigma^2)$ |
| GJR-GARCH | Assimetria via indicadora |
| APARCH | Potencia flexivel + assimetria |
| IGARCH | Persistencia unitaria |
| CGARCH | Decomposicao curto/longo prazo |

In [ ]:
# Load data
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns'].values

# Estimate all 6 models
models = {}
results = {}

model_specs = {
    'GARCH': lambda r: GARCH(r, p=1, q=1),
    'EGARCH': lambda r: EGARCH(r, p=1, q=1),
    'GJR': lambda r: GJRGARCH(r, p=1, q=1),
    'APARCH': lambda r: APARCH(r, p=1, q=1),
    'IGARCH': lambda r: IGARCH(r),
    'CGARCH': lambda r: ComponentGARCH(r),
}

for name, model_fn in model_specs.items():
    print(f"Estimating {name}...")
    models[name] = model_fn(returns)
    results[name] = models[name].fit(disp=False)
    print(f"  LogLik: {results[name].loglike:.4f}, AIC: {results[name].aic:.4f}, "
          f"Params: {len(results[name].params)}")

print("\n" + "=" * 70)
print("All 6 models estimated successfully!")
print("=" * 70)

# Print summary for each model
for name, res in results.items():
    print(f"\n{'='*70}")
    print(f"  {name}")
    print(f"{'='*70}")
    print(res.summary())

## 2. Criterios de informacao

Os criterios de informacao balanceiam **ajuste** (log-verossimilhanca) com **parcimonia** (numero de parametros):

$$\text{AIC} = -2\ell + 2k$$
$$\text{BIC} = -2\ell + k \ln(n)$$
$$\text{HQIC} = -2\ell + 2k \ln(\ln(n))$$

O BIC penaliza mais fortemente modelos com muitos parametros, sendo mais conservador
na selecao de modelos. Em amostras grandes ($n > 100$), $\ln(n) > 2$, entao BIC > AIC.

**Regra**: menor valor = melhor modelo.

In [ ]:
# Information criteria comparison table
ic_table = pd.DataFrame({
    name: {
        'AIC': res.aic,
        'BIC': res.bic,
        'HQIC': res.hqic,
        'LogLik': res.loglike,
        'Params': len(res.params),
    }
    for name, res in results.items()
}).T

print("Information Criteria Comparison")
print("=" * 70)
print(ic_table.to_string())

# Sort by AIC
print("\n\nRanking by AIC (lower is better):")
print(ic_table.sort_values('AIC')[['AIC', 'BIC', 'HQIC']].to_string())

# Identify best model per criterion
print("\n")
for criterion in ['AIC', 'BIC', 'HQIC']:
    best = ic_table[criterion].astype(float).idxmin()
    print(f"Best by {criterion}: {best} ({ic_table.loc[best, criterion]:.4f})")

# Visualize with bar chart
plot_model_comparison(results)
plt.show()

## 3. Diagnosticos dos residuos

Um modelo bem especificado deve produzir **residuos padronizados** ($z_t = \epsilon_t / \sigma_t$)
que sejam i.i.d. Em particular:

- **Teste ARCH-LM** nos residuos: se p-valor > 0.05, o modelo capturou os efeitos ARCH
- **Ljung-Box** nos residuos ao quadrado ($z_t^2$): testa se ha autocorrelacao remanescente

Se os residuos passam nesses testes, o modelo esta **adequadamente especificado**.
Se falham, ha dependencia nao capturada — considere um modelo mais complexo.

In [ ]:
# Diagnostic tests: ARCH-LM and Ljung-Box on standardized residuals
diag_rows = []

for name, res in results.items():
    resids = res.resid
    arch_test = arch_lm_test(resids, lags=10)
    lb_test = ljung_box_squared(resids, lags=10)

    diag_rows.append({
        'Model': name,
        'ARCH-LM Stat': arch_test.statistic,
        'ARCH-LM p-val': arch_test.pvalue,
        'ARCH-LM': 'PASS' if arch_test.pvalue > 0.05 else 'FAIL',
        'LB Stat': lb_test.statistic,
        'LB p-val': lb_test.pvalue,
        'LB': 'PASS' if lb_test.pvalue > 0.05 else 'FAIL',
    })

diag_table = pd.DataFrame(diag_rows).set_index('Model')

print("Residual Diagnostics (lags=10)")
print("=" * 80)
print(diag_table.to_string())

# Summary
print("\n\nModels passing all diagnostic tests:")
passing = diag_table[(diag_table['ARCH-LM'] == 'PASS') & (diag_table['LB'] == 'PASS')]
if len(passing) > 0:
    for name in passing.index:
        print(f"  - {name}")
else:
    print("  None (all models have some residual dependence)")

# Visualize p-values
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.35
ax.bar(x - width/2, diag_table['ARCH-LM p-val'], width, label='ARCH-LM p-value', color='steelblue')
ax.bar(x + width/2, diag_table['LB p-val'], width, label='Ljung-Box p-value', color='darkorange')
ax.axhline(0.05, color='red', linestyle='--', linewidth=1, label='5% significance level')
ax.set_xticks(x)
ax.set_xticklabels(diag_table.index, rotation=30, ha='right')
ax.set_ylabel('p-value')
ax.set_title('Diagnostic Test p-values by Model')
ax.legend()
fig.tight_layout()
plt.show()

## 4. Previsao out-of-sample

Criterios de informacao avaliam o ajuste **in-sample**. Para avaliar a capacidade
**preditiva**, usamos a estrategia de **rolling window** (janela deslizante):

1. Defina uma janela de estimacao (ex: 1000 obs) e um horizonte de previsao (ex: 1 dia)
2. Estime o modelo na janela $[t-W+1, t]$
3. Faca previsao de $\sigma^2_{t+1}$
4. Compare com o proxy de variancia realizada: $r_{t+1}^2$
5. Deslize a janela 1 dia e repita

**Metrica**: RMSE (Root Mean Squared Error) da previsao:

$$\text{RMSE} = \sqrt{\frac{1}{T_{oos}} \sum_{t=1}^{T_{oos}} (\hat{\sigma}^2_{t+1} - r_{t+1}^2)^2}$$

In [ ]:
# Rolling window out-of-sample forecast evaluation
window_size = 1000
step = 20  # use step > 1 to save computation time
n_total = len(returns)
forecast_indices = range(window_size, n_total, step)
n_forecasts = len(list(forecast_indices))

print("Rolling window forecast evaluation")
print(f"Window size: {window_size}, Step: {step}, Total forecasts: {n_forecasts}")
print("=" * 60)

# Store squared forecast errors for each model
forecast_errors = {name: [] for name in model_specs}

for t in forecast_indices:
    window = returns[t - window_size:t]
    actual_var = returns[t] ** 2  # realized variance proxy

    for name, model_fn in model_specs.items():
        try:
            m = model_fn(window)
            r = m.fit(disp=False)
            fcast = r.forecast(horizon=1)
            predicted_var = fcast['variance'][0]
            forecast_errors[name].append((predicted_var - actual_var) ** 2)
        except Exception:
            forecast_errors[name].append(np.nan)

# Compute RMSE for each model
rmse_dict = {}
for name in model_specs:
    errors = np.array(forecast_errors[name])
    valid = errors[~np.isnan(errors)]
    rmse_dict[name] = np.sqrt(np.mean(valid)) if len(valid) > 0 else np.nan

rmse_series = pd.Series(rmse_dict, name='RMSE')
print("\nOut-of-Sample RMSE (Variance Forecast)")
print("-" * 40)
for name, rmse in rmse_series.sort_values().items():
    print(f"  {name:<10} RMSE: {rmse:.6e}")

# Plot RMSE comparison
fig, ax = plt.subplots(figsize=(8, 5))
rmse_series.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('RMSE')
ax.set_title('Out-of-Sample Forecast RMSE (Rolling Window)')
fig.tight_layout()
plt.show()

## 5. Ranking final

Vamos combinar todos os criterios para um **ranking final** dos modelos:

1. **AIC** (ajuste in-sample com penalizacao leve)
2. **BIC** (ajuste in-sample com penalizacao forte)
3. **ARCH-LM p-valor** (adequacao dos residuos)
4. **RMSE out-of-sample** (capacidade preditiva)

Para cada criterio, atribuimos um **rank** (1 = melhor, 6 = pior).
O ranking final e a **media dos ranks**.

In [ ]:
# Final ranking combining in-sample and out-of-sample criteria

# Build ranking DataFrame
rank_data = pd.DataFrame({
    'AIC': ic_table['AIC'].astype(float),
    'BIC': ic_table['BIC'].astype(float),
    'ARCH-LM p-val': diag_table['ARCH-LM p-val'].astype(float),
    'RMSE': rmse_series,
})

# Compute ranks (lower AIC/BIC/RMSE is better, higher p-value is better)
rank_table = pd.DataFrame(index=rank_data.index)
rank_table['AIC_rank'] = rank_data['AIC'].rank()
rank_table['BIC_rank'] = rank_data['BIC'].rank()
rank_table['ARCHLM_rank'] = rank_data['ARCH-LM p-val'].rank(ascending=False)  # higher is better
rank_table['RMSE_rank'] = rank_data['RMSE'].rank()

# Mean rank
rank_table['Mean_rank'] = rank_table.mean(axis=1)

# Sort by mean rank
rank_table = rank_table.sort_values('Mean_rank')

print("Final Model Ranking")
print("=" * 70)
print("(Rank 1 = best, 6 = worst for each criterion)")
print()
print(rank_table.to_string(float_format='%.2f'))

# Summary with original values alongside ranks
summary = pd.DataFrame({
    'AIC': rank_data['AIC'].apply(lambda x: f'{x:.2f}'),
    'BIC': rank_data['BIC'].apply(lambda x: f'{x:.2f}'),
    'ARCH-LM p': rank_data['ARCH-LM p-val'].apply(lambda x: f'{x:.4f}'),
    'RMSE': rank_data['RMSE'].apply(lambda x: f'{x:.2e}'),
    'Mean Rank': rank_table['Mean_rank'].apply(lambda x: f'{x:.2f}'),
})
summary = summary.loc[rank_table.index]

print("\n\nFinal Summary (sorted by mean rank)")
print("=" * 70)
print(summary.to_string())

best_model = rank_table.index[0]
print(f"\n{'='*70}")
print(f"BEST MODEL: {best_model} (Mean Rank: {rank_table.loc[best_model, 'Mean_rank']:.2f})")
print(f"{'='*70}")

# Visualization: radar-style comparison using bar chart
fig, ax = plt.subplots(figsize=(10, 6))
rank_table.drop(columns='Mean_rank').plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Model Rankings by Criterion (lower is better)')
ax.set_ylabel('Rank')
ax.set_xlabel('Model')
ax.legend(title='Criterion', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
fig.tight_layout()
plt.show()

## 6. Conclusoes e recomendacoes praticas

### Guia de selecao de modelos GARCH

| Situacao | Modelo recomendado | Justificativa |
|----------|-------------------|---------------|
| **Baseline simples** | GARCH(1,1) | Robusto, poucos parametros, facil de interpretar |
| **Efeito alavancagem evidente** | GJR-GARCH ou EGARCH | Capturam assimetria com 1 parametro extra |
| **Potencia otima desconhecida** | APARCH | Deixa os dados determinarem $\delta$ |
| **Persistencia muito alta** | IGARCH | Formaliza $\alpha + \beta = 1$ |
| **Tendencia na volatilidade** | Component-GARCH | Separa curto e longo prazo |
| **Amostra pequena** | GARCH(1,1) | Evita sobreparametrizacao |

### Recomendacoes gerais

1. **Sempre comece pelo GARCH(1,1)** — e o benchmark universal
2. **Teste efeito alavancagem** com o Sign Bias test antes de adotar modelos assimetricos
3. **Use BIC para selecao** se a amostra e grande (BIC e consistente)
4. **Valide com diagnosticos** — o melhor AIC nao garante residuos limpos
5. **Avalie out-of-sample** quando o objetivo e previsao